# CYBR 3570 - Crypto Lab 01
## Why Modern Cryptography Exists

**Week:** 1  
**Toolkit Version:** v0.1  
**Big Question:** Why can't we just invent our own encryption algorithm?

This notebook is part lecture notes, part lab manual, and part assignment. By the end, you will have started your semester-long `crypto_toolkit` package.

## Learning Objectives

By the end of this notebook, you should be able to:

1. Explain what cryptography is and what problems it can help solve.
2. Distinguish plaintext, ciphertext, keys, and key space.
3. Implement a Caesar cipher for educational purposes.
4. Break a Caesar cipher using brute force.
5. Perform a simple frequency analysis on ciphertext.
6. Explain why classical ciphers fail against modern attackers.
7. Connect these failures to the professional rule: **do not roll your own production cryptography**.

## Today's Big Question

> Why can't we just invent our own encryption algorithm?

You do not need to answer this fully yet. We will return to it at the end of the lab.

## Why This Matters

Imagine you are building a messaging feature for a web application. A user types:

> Meet me outside PKI after class.

Before the message reaches the recipient, it may pass through networks, routers, logs, servers, storage systems, backups, and monitoring tools. Cryptography can help protect the message, but only if the correct cryptographic tools are used in the correct way.

Classical ciphers give us a safe place to observe failure. We can break them quickly, learn what went wrong, and use those lessons to understand why modern cryptography exists.

## Reading Check

Answer briefly in Markdown.

1. What is the difference between cryptography and cryptanalysis?
2. What is the difference between plaintext and ciphertext?
3. What is a key space?
4. What does Kerckhoffs' Principle say?
5. Why is a large key space not always enough to make a cipher secure?

## Setup Check

Run the following cell. If it fails, your Python environment is not ready yet.

In [1]:
import sys
from pathlib import Path
from collections import Counter

print(sys.version)
print(Path.cwd())

3.14.3 (tags/v3.14.3:323c59a, Feb  3 2026, 16:04:56) [MSC v.1944 64 bit (AMD64)]
c:\Users\user\Downloads\CYBR-3570-main\CYBR-3570-main\crypto-toolkit-week01-starter\notebooks


## Part 1 - Representing Letters as Numbers

The Caesar cipher treats letters as numbers:

| Letter | A | B | C | ... | Z |
|---|---:|---:|---:|---|---:|
| Number | 0 | 1 | 2 | ... | 25 |

Encryption with shift `k` is:

$$E_k(x) = (x + k) \mod 26$$

Decryption is:

$$D_k(y) = (y - k) \mod 26$$

In [2]:
ALPHABET = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"

def char_to_num(ch: str) -> int:
    """Convert uppercase A-Z to a number 0-25."""
    return ALPHABET.index(ch)


def num_to_char(n: int) -> str:
    """Convert a number to uppercase A-Z using modulo 26."""
    return ALPHABET[n % 26]

print(char_to_num("A"), char_to_num("Z"))
print(num_to_char(0), num_to_char(25), num_to_char(26))

0 25
A Z A


## Part 2 - Guided Caesar Encryption

Complete the function below. Preserve non-alphabetic characters so messages remain readable.

In [3]:
def caesar_encrypt(plaintext: str, shift: int) -> str:
    """
    Encrypt plaintext using the Caesar cipher.

    This is an educational implementation only.
    Do not use this algorithm to protect real information.
    """
    result = []
    for ch in plaintext:
        if ch.upper() in ALPHABET:
            original_is_lower = ch.islower()
            n = char_to_num(ch.upper())
            encrypted = num_to_char(n + shift)
            result.append(encrypted.lower() if original_is_lower else encrypted)
        else:
            result.append(ch)
    return "".join(result)

message = "Meet me at PKI after class."
ciphertext = caesar_encrypt(message, 3)
print(ciphertext)

Phhw ph dw SNL diwhu fodvv.


## Part 3 - Decryption

Because the Caesar cipher is based on modular addition, decryption can reuse the encryption function with the negative shift.

In [4]:
def caesar_decrypt(ciphertext: str, shift: int) -> str:
    """Decrypt Caesar ciphertext by reversing the shift."""
    return caesar_encrypt(ciphertext, -shift)

print(caesar_decrypt(ciphertext, 3))

Meet me at PKI after class.


## Checkpoint 1

In the cell below, encrypt your own message using a shift of your choice. Then decrypt it.

In [5]:
my_message = "Cybersecurity is interesting."
my_shift = 5

my_ciphertext = caesar_encrypt(my_message, my_shift)
my_plaintext = caesar_decrypt(my_ciphertext, my_shift)

print(my_ciphertext)
print(my_plaintext)

Hdgjwxjhzwnyd nx nsyjwjxynsl.
Cybersecurity is interesting.


## Part 4 - Brute Force Attack

The Caesar cipher has only 26 possible keys. An attacker can try all of them.

In [ ]:
def brute_force_caesar(ciphertext: str) -> list[tuple[int, str]]:
    """Return all possible Caesar decryptions."""
    candidates = []
    for shift in range(26):
        candidates.append((shift, caesar_decrypt(ciphertext, shift)))
    return candidates

secret = caesar_encrypt("THE KEY SPACE IS TOO SMALL", 11)
for shift, candidate in brute_force_caesar(secret):
    print(f"shift={shift:2d}: {candidate}")

## Checkpoint 2

In your own words: why does brute force work so well against Caesar?

**Answer:**  
Brute force works well against Caesar because there are only 26 possible shifts. An attacker can easily try every shift until they find a message that makes sense.

## Part 5 - Frequency Analysis

Many classical ciphers preserve patterns from the original language. English text does not use all letters equally. If a cipher preserves those patterns, it leaks information.

In [ ]:
def clean_letters(text: str) -> str:
    """Return only uppercase A-Z characters."""
    return "".join(ch for ch in text.upper() if ch in ALPHABET)


def letter_frequencies(text: str) -> dict[str, float]:
    """Compute relative letter frequencies for A-Z."""
    letters = clean_letters(text)
    counts = Counter(letters)
    total = len(letters)
    if total == 0:
        return {ch: 0.0 for ch in ALPHABET}
    return {ch: counts.get(ch, 0) / total for ch in ALPHABET}

sample = "THIS IS A SAMPLE MESSAGE WITH SEVERAL LETTERS AND SOME REPEATED LETTERS"
freq = letter_frequencies(sample)
print(sorted(freq.items(), key=lambda item: item[1], reverse=True)[:5])

In [ ]:
import matplotlib.pyplot as plt

long_plaintext = """
Cryptography is everywhere in modern computing. It protects communication,
software updates, passwords, payment systems, and private data. Weak cryptography
can fail even when the program appears to work correctly.
"""
long_ciphertext = caesar_encrypt(long_plaintext, 9)

freq_plain = letter_frequencies(long_plaintext)
freq_cipher = letter_frequencies(long_ciphertext)

plt.figure(figsize=(10, 4))
plt.bar(freq_cipher.keys(), freq_cipher.values())
plt.title("Letter Frequencies in Caesar Ciphertext")
plt.xlabel("Letter")
plt.ylabel("Relative Frequency")
plt.show()

## Checkpoint 3

Look at the frequency graph. What information about the plaintext is still visible in the ciphertext?

The ciphertext still shows patterns from the original message. Some letters appear more often than others, so an attacker can use those frequencies to help figure out the plaintext.




## Part 6 - Toolkit Integration

Now move the Caesar functions into your toolkit.

Create or update this file:

```text
crypto_toolkit/classical/caesar.py
```

At minimum, it should include:

- `caesar_encrypt`
- `caesar_decrypt`
- `brute_force_caesar`

Add a module-level warning that the implementation is educational and insecure.

In [ ]:
# Optional helper: create starter folders from inside the notebook.
for folder in [
    Path("crypto_toolkit"),
    Path("crypto_toolkit/classical"),
    Path("tests"),
]:
    folder.mkdir(parents=True, exist_ok=True)

for init_file in [Path("crypto_toolkit/__init__.py"), Path("crypto_toolkit/classical/__init__.py")]:
    init_file.touch(exist_ok=True)

print("Toolkit folders ready.")

## Part 7 - Minimal Tests

Testing is a professional habit. Cryptographic software especially needs careful testing.

Create or update:

```text
tests/test_caesar.py
```

Use these examples as a starting point.

In [ ]:
# These are examples you can place in tests/test_caesar.py

def test_caesar_round_trip():
    msg = "Attack at dawn!"
    shift = 5
    assert caesar_decrypt(caesar_encrypt(msg, shift), shift) == msg


def test_caesar_known_value():
    assert caesar_encrypt("ABC XYZ", 3) == "DEF ABC"

# Run these inside the notebook for a quick check.
test_caesar_round_trip()
test_caesar_known_value()
print("Notebook tests passed.")

## Vigenere Cipher Extention

A Vigenere Cipher uses a keyword to determine a the amount of shift. For example the plaintext ***Cryptography is fun*** with a keyword **abcde** would shift the **C** by 0 letters (since A is the 0th letter of the alphabet). Then the **r** would shift by 1, the **y** by 2, etc. since these shifts correspond to the location in the alphabet of each letter of the keyword. Once the keyword is complete, the shift restarts at the beginning.

### Implement a Vigenere Encode/Decode
- Look at the Vigenere.py in the crypto_toolkit/classical
- Use the themes of a Caesar Shift to build this cipher
- How could we look at ciphertext and determine a possible length for the keyword?

## Security Engineering Discussion

Answer in Markdown.

1. If Caesar is so easy to break, why is it useful in this course?
2. What did brute force reveal about key space?
3. What did frequency analysis reveal about statistical leakage?
4. How do these failures support Kerckhoffs' Principle?
5. What should you do in a real project if you need encryption?

1. Caesar is useful because it makes it easy to learn the basic ideas of encryption and decryption.

2. Brute force shows that a small key space is weak because an attacker can quickly try every possible key.

3. Frequency analysis shows that ciphertext can still leak patterns about the original message, such as how often certain letters appear.

4. These failures support Kerckhoffs' Principle because an encryption system should stay secure even if an attacker knows how the algorithm works. Security should depend on the key, not keeping the algorithm secret.

5. In a real project, I should use trusted and tested encryption libraries instead of creating my own encryption algorithm.

## Engineering Log Entry

Add a short entry to `ENGINEERING_LOG.md`.

Suggested format:

```markdown
## Week 1 - Toolkit v0.1

### Added
- Caesar cipher educational implementation
- Brute force demonstration
- Frequency analysis helper

### Security Lesson
Classical ciphers are useful for learning, but they should never be used to protect real data.

### Reflection
...
```

## Final Reflection: Return to the Big Question

> Why can't we just invent our own encryption algorithm?

We can't just invent our own encryption algorithm because something that looks secure can still have weaknesses. In this lab, Caesar was easy to break because there were only 26 possible shifts, so brute force could try every key. Frequency analysis also showed that ciphertext can still reveal patterns from the original message. This showed me why real projects should use encryption methods that have already been tested instead of making their own.

**Final Reflection:**  
Write your response here.

## Submission Checklist

Before submitting, confirm that you have:

- [yes ] Completed the reading check.
- [ yes] Implemented Caesar encryption and decryption.
- [ yes] Demonstrated brute force decryption.
- [ yes] Completed frequency analysis.
- [yes ] Added toolkit files under `crypto_toolkit/classical/`.
- [ yes] Added or updated tests.
- [ yes] Added an engineering log entry.
- [ yes] Committed and pushed your work.